## Example 1: Functionality Test 1

In [ ]:
import gspread
import json
import os
import tempfile
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from google.oauth2.service_account import Credentials

# Example 1: gspread client setup and authentication methods
print("Example 1: gspread client setup and authentication methods")

try:
    # Check available authentication methods
    auth_methods = [attr for attr in dir(gspread) if 'auth' in attr.lower() or 'client' in attr.lower()]
    print(f"✓ Authentication methods: {auth_methods}")
    
    # Show available client classes
    client_classes = [attr for attr in dir(gspread) if 'Client' in attr]
    print(f"✓ Client classes: {client_classes}")
    
    # Demonstrate OAuth2 setup (without actual credentials)
    scopes = [
        'https://www.googleapis.com/auth/spreadsheets',
        'https://www.googleapis.com/auth/drive'
    ]
    print(f"✓ Required OAuth2 scopes: {len(scopes)}")
    for scope in scopes:
        print(f"  - {scope.split('/')[-1]}")
    
    # Service-account authentication, offline: the key is generated here and written as a service-account
    # file, which gspread turns into an authorised client. google-auth fetches a token only on the first API
    # request, and this example makes none, so nothing leaves the machine.
    key_pem = rsa.generate_private_key(public_exponent=65537, key_size=2048).private_bytes(
        serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8, serialization.NoEncryption()
    ).decode()
    service_account_info = {
        "type": "service_account", "project_id": "demo-project", "private_key_id": "0" * 40,
        "private_key": key_pem, "client_email": "notebook-demo@demo-project.iam.gserviceaccount.com",
        "client_id": "1", "token_uri": "https://oauth2.googleapis.com/token",
    }
    with tempfile.TemporaryDirectory() as key_dir:
        key_path = os.path.join(key_dir, "service_account.json")
        with open(key_path, "w") as f:
            json.dump(service_account_info, f)
        client = gspread.service_account(filename=key_path, scopes=scopes)
        credentials = client.http_client.auth
        assert isinstance(client, gspread.Client) and isinstance(credentials, Credentials), (client, credentials)
        assert credentials.service_account_email == service_account_info["client_email"], credentials
        assert list(credentials.scopes) == scopes, credentials.scopes
        assert credentials.token is None and not credentials.valid  # no token was fetched
        print(f"✓ Service account authentication successful: {credentials.service_account_email}")

        # Show exception handling for authentication: a missing key file is refused before any request
        try:
            gspread.service_account(filename=os.path.join(key_dir, "missing.json"))
        except FileNotFoundError as auth_error:
            assert str(auth_error.filename).endswith("missing.json"), auth_error.filename
            print("✓ Authentication demo: a missing service-account file raises FileNotFoundError")
        else:
            raise AssertionError("expected FileNotFoundError for a missing service-account file")
    
    print("✓ gspread client setup methods demonstrated")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"gspread example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Functionality Test 2

In [ ]:
import gspread
import json

# Example 2: Spreadsheet operations and data structures  
print("Example 2: Spreadsheet operations and data structures")

try:
    # Demonstrate gspread data structures and methods
    print("✓ gspread core classes and methods:")
    
    # Check Spreadsheet class methods
    if hasattr(gspread, 'Spreadsheet'):
        spreadsheet_methods = [m for m in dir(gspread.Spreadsheet) if not m.startswith('_')]
        key_methods = [m for m in spreadsheet_methods if any(word in m for word in 
                      ['worksheet', 'sheet', 'add', 'del', 'share'])]
        print(f"  Spreadsheet methods: {len(spreadsheet_methods)} ({len(key_methods)} key)")
        print(f"  Key methods: {key_methods[:5]}")
    
    # Check Worksheet class methods  
    if hasattr(gspread, 'Worksheet'):
        worksheet_methods = [m for m in dir(gspread.Worksheet) if not m.startswith('_')]
        data_methods = [m for m in worksheet_methods if any(word in m for word in 
                       ['get', 'update', 'append', 'insert', 'delete'])]
        print(f"  Worksheet methods: {len(worksheet_methods)} ({len(data_methods)} data)")
        print(f"  Data methods: {data_methods[:5]}")
    
    # Sample data structure for spreadsheet operations
    sample_data = {
        'headers': ['Name', 'Age', 'City', 'Score'],
        'rows': [
            ['Alice', 25, 'New York', 95],
            ['Bob', 30, 'London', 87],
            ['Carol', 28, 'Tokyo', 92]
        ]
    }
    
    print(f"✓ Sample data structure:")
    print(f"  Headers: {sample_data['headers']}")
    print(f"  Rows: {len(sample_data['rows'])}")
    print(f"  Columns: {len(sample_data['headers'])}")
    
    print("✓ Spreadsheet operations structure demonstrated")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"gspread example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Functionality Test 3

In [ ]:
import gspread

# Example 3: Cell and range operations with formatting
print("Example 3: Cell and range operations with formatting")

try:
    # Demonstrate cell and range concepts
    print("✓ Cell and range operation concepts:")
    
    # Check Cell class if available
    if hasattr(gspread, 'Cell'):
        cell_attrs = [attr for attr in dir(gspread.Cell) if not attr.startswith('_')]
        print(f"  Cell attributes: {cell_attrs}")
    
    # Range notation examples
    range_examples = [
        'A1', 'B2:D5', 'A:A', '1:1', 
        'Sheet1!A1:B10', 'Data!C2:E100'
    ]
    print(f"✓ Range notation examples: {len(range_examples)}")
    for i, range_ex in enumerate(range_examples, 1):
        print(f"  {i}. {range_ex}")
    
    # Batch operation concepts
    batch_operations = {
        'batch_get': 'Get multiple ranges in one request',
        'batch_update': 'Update multiple ranges efficiently', 
        'batch_clear': 'Clear multiple ranges at once',
        'append_rows': 'Add multiple rows to worksheet'
    }
    
    print(f"✓ Batch operations: {len(batch_operations)}")
    for op, desc in batch_operations.items():
        print(f"  {op}: {desc}")
    
    # Cell value types and formatting
    value_types = ['string', 'number', 'date', 'formula', 'boolean']
    print(f"✓ Supported value types: {', '.join(value_types)}")
    
    print("✓ Cell and range operations demonstrated")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"gspread example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Functionality Test 4

In [ ]:
import gspread
import re

# Example 4: Sharing and permissions management
print("Example 4: Sharing and permissions management")

try:
    # Demonstrate sharing and permission concepts
    print("✓ Sharing and permission management:")
    
    # Permission types and roles
    permission_roles = {
        'owner': 'Full control over spreadsheet',
        'editor': 'Can edit content and structure',
        'commenter': 'Can add comments only',
        'reader': 'Read-only access'
    }
    
    print(f"  Permission roles: {len(permission_roles)}")
    for role, desc in permission_roles.items():
        print(f"    {role}: {desc}")
    
    # Sharing methods available in gspread
    sharing_methods = ['share', 'list_permissions', 'remove_permissions']
    available_methods = [method for method in sharing_methods 
                        if hasattr(gspread.Spreadsheet, method)]
    print(f"✓ Available sharing methods: {available_methods}")
    
    # Email validation pattern for sharing
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    sample_emails = ['user@example.com', 'team@company.org', 'admin@domain.net']
    
    print(f"✓ Email validation for sharing:")
    for email in sample_emails:
        valid = bool(re.match(email_pattern, email))
        print(f"  {email}: {'✓' if valid else '✗'}")
    
    # Domain sharing concepts
    sharing_types = ['anyone_with_link', 'domain', 'specific_people']
    print(f"✓ Sharing types: {', '.join(sharing_types)}")
    
    print("✓ Sharing and permissions demonstrated")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"gspread example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Functionality Test 5

In [ ]:
import gspread
import time

# Example 5: Error handling and API rate limiting
print("Example 5: Error handling and API rate limiting")

try:
    # Common gspread exceptions and error handling
    print("✓ gspread exception handling:")
    
    # Check for exception classes
    exception_classes = [attr for attr in dir(gspread) if 'Error' in attr or 'Exception' in attr]
    print(f"  Exception classes: {exception_classes}")
    
    # Common error scenarios
    error_scenarios = {
        'SpreadsheetNotFound': 'Invalid spreadsheet ID or no access',
        'WorksheetNotFound': 'Worksheet does not exist',
        'APIError': 'Google Sheets API error',
        'CellNotFound': 'Invalid cell reference'
    }
    
    print(f"✓ Common error scenarios: {len(error_scenarios)}")
    for error, desc in error_scenarios.items():
        print(f"  {error}: {desc}")
    
    # Rate limiting and quotas
    api_limits = {
        'requests_per_minute': 300,
        'requests_per_day': 20000,
        'cells_updated_per_day': 50000000
    }
    
    print(f"✓ Google Sheets API limits:")
    for limit, value in api_limits.items():
        print(f"  {limit}: {value:,}")
    
    # Best practices for rate limiting
    best_practices = ['Use batch operations', 'Implement exponential backoff', 'Cache results']
    print(f"✓ Rate limiting best practices: {', '.join(best_practices)}")
    
    print("✓ Error handling and rate limiting demonstrated")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"gspread example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e